# Protein Clustering Analysis with TSFEL Features

This notebook performs clustering analysis on protein features extracted using TSFEL.

## Setup Instructions

1. **Set up environment variable** (choose one method):
   ```bash
   # Option 1: Set in terminal before running
   export ML4NP_DATA_ROOT="/path/to/your/data"
   
   # Option 2: Create a .env file in notebook directory
   echo "ML4NP_DATA_ROOT=/path/to/your/data" > .env
   ```

2. **Expected data structure**:
   ```
   /path/to/your/data/
   └── complete_feature_sets/
       └── tsfel.csv  # TSFEL feature file
   ```

3. **Install dependencies**:
   ```bash
   pip install pandas numpy scikit-learn umap-learn seaborn matplotlib plotly xgboost shap tsfel hdbscan
   ```

In [ ]:
# Configuration
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load data root from environment variable
DATA_ROOT = Path(os.environ.get("ML4NP_DATA_ROOT", "./data"))

# Define paths
TSFEL_FEATURES_PATH = DATA_ROOT / "tsfel.csv"

# Create output directory
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Configuration parameters
CONFIG = {
    "nan_threshold": 0.5,  # Drop columns with >50% NaN values
    "variance_threshold": 0.001,  # Minimum variance for feature selection
    "outlier_contamination": 0.3,  # IsolationForest contamination parameter
    "n_clusters": 4,  # Number of clusters for KMeans
    "sparsity_s": 8.6,  # Sparsity parameter for sparse KMeans
    "random_state": 42,  # Random seed for reproducibility
    "umap_n_components": 2,  # UMAP dimensionality
}

print(f"Data root: {DATA_ROOT}")
print(f"TSFEL features path: {TSFEL_FEATURES_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Configuration: {CONFIG}")

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.optimize import linear_sum_assignment, bisect
from scipy import stats

# Scikit-learn imports
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold, f_classif
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score, confusion_matrix,
    precision_score, recall_score, f1_score,
    precision_recall_fscore_support, silhouette_score,
    homogeneity_completeness_v_measure,
    calinski_harabasz_score, davies_bouldin_score,
    normalized_mutual_info_score
)

# Other imports
import umap
import xgboost as xgb
import shap
import hdbscan
import tsfel

# Set style for plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("tab10")
print("All imports completed successfully")

## 1. Data Loading and Preprocessing

In [ ]:
def load_and_preprocess_data(filepath):
    """Load and preprocess the TSFEL feature data."""
    print(f"Loading data from {filepath}")
    
    if not filepath.exists():
        raise FileNotFoundError(f"Feature file not found at {filepath}. "
                              f"Please ensure ML4NP_DATA_ROOT is set correctly.")
    
    # Load data
    df = pd.read_csv(filepath)
    print(f"Loaded data shape: {df.shape}")
    
    # Ensure label column exists
    if 'label' not in df.columns:
        raise ValueError("Data must contain 'label' column")
    
    # Step 1: Drop features with >50% NaN values
    nan_threshold = CONFIG["nan_threshold"] * len(df)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df = df.dropna(axis=1, thresh=nan_threshold)
    print(f"After NaN thresholding: {df.shape}")
    
    # Step 2: Drop low-variance features
    numeric_features = df.select_dtypes(include=['number']).columns.tolist()
    if 'label' in numeric_features:
        numeric_features.remove('label')
    
    selector = VarianceThreshold(threshold=CONFIG["variance_threshold"])
    reduced_features = selector.fit_transform(df[numeric_features])
    retained_columns = df[numeric_features].columns[selector.get_support()]
    df = df[retained_columns.tolist() + ['label']]
    print(f"After variance thresholding: {df.shape}")
    
    # Step 3: Fill remaining NaNs with median
    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    df = df.dropna()
    
    # Step 4: Remove outliers using IsolationForest per label
    df = remove_outliers_by_label(df, label_column='label', 
                                  contamination=CONFIG["outlier_contamination"])
    
    # Step 5: Unite all peptide ladders to single label
    df["label"] = df["label"].apply(lambda x: 'ladder' if 'ladder' in str(x).lower() else x)
    
    # Step 6: Drop MFCC columns if present
    mfcc_cols = [col for col in df.columns if 'MFCC' in col]
    if mfcc_cols:
        df = df.drop(columns=mfcc_cols)
    
    print(f"Final preprocessed data shape: {df.shape}")
    print(f"Label distribution:\n{df['label'].value_counts()}")
    
    return df

def remove_outliers_by_label(df, label_column, contamination=0.3):
    """Remove outliers within each label using IsolationForest."""
    cleaned_df = pd.DataFrame()
    
    for label in df[label_column].unique():
        subset = df[df[label_column] == label]
        if len(subset) > 1:  # Need at least 2 samples for outlier detection
            iso_forest = IsolationForest(contamination=contamination, random_state=CONFIG["random_state"])
            outliers = iso_forest.fit_predict(subset.drop(columns=[label_column]))
            subset_cleaned = subset[outliers == 1]
        else:
            subset_cleaned = subset
        
        cleaned_df = pd.concat([cleaned_df, subset_cleaned], ignore_index=True)
    
    return cleaned_df

In [ ]:
# Load and preprocess data
final_feature_df = load_and_preprocess_data(TSFEL_FEATURES_PATH)

## 2. Feature Analysis

In [ ]:
def analyze_features(df):
    """Analyze feature correlations and importance."""
    
    # Select a subset of features for correlation analysis
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    sample_cols = numeric_cols[:15]  # First 15 numeric features
    
    if len(sample_cols) >= 2:
        # Correlation matrix
        corr_matrix = df[sample_cols].corr(method='pearson')
        
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", 
                    square=True, cbar_kws={"shrink": 0.8})
        plt.title("Feature Correlation Matrix (Sample)")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "feature_correlation.png", dpi=300, bbox_inches='tight')
        plt.show()
    
    # ANOVA F-statistics for feature importance
    X = df.drop(columns=['label'])
    y = df['label']
    
    if len(X.columns) > 0:
        f_values, p_values = f_classif(X, y)
        
        # Select top features
        top_n = min(10, len(X.columns))
        top_indices = np.argsort(-f_values)[:top_n]
        top_features = X.columns[top_indices]
        
        print(f"Top {top_n} decisive features:")
        for i, feat in enumerate(top_features):
            print(f"  {i+1}. {feat}: F={f_values[top_indices[i]]:.2f}, p={p_values[top_indices[i]]:.4f}")
        
        # Boxplot of top features
        plt.figure(figsize=(12, 6))
        df_melted = df.melt(id_vars=['label'], value_vars=top_features[:5])
        sns.boxplot(x="variable", y="value", hue="label", data=df_melted)
        plt.xticks(rotation=45)
        plt.title("Top Feature Distributions Across Labels")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "feature_distributions.png", dpi=300, bbox_inches='tight')
        plt.show()
    
    return X, y

In [ ]:
X, y = analyze_features(final_feature_df)

## 3. Mean-Related Feature Removal (Based on Domain Resarch from Documentation)

In [ ]:
def prepare_features_for_clustering(df):
    """Prepare features by removing correlated and low-importance features."""
    
    # List of features to drop based on correlation and domain knowledge
    features_to_drop = [
        '0_ECDF Percentile Count_0', '0_Interquartile range',
        '0_ECDF Percentile Count_1', '0_ECDF Percentile_0',
        '0_ECDF Percentile_1', '0_ECDF_3', '0_ECDF_4',
        '0_ECDF_5', '0_ECDF_6', '0_ECDF_7', '0_ECDF_8',
        '0_ECDF_9', '0_Min', '0_Max', '0_Median frequency',
        '0_Centroid', '0_Spectral centroid', '0_Standard deviation',
        '0_Variance', '0_Median frequency', "0_Root mean square",
        "0_Average power", "0_Histogram mode", "0_Mean", "0_Median"
    ]
    
    # Only drop features that actually exist in the dataframe
    existing_features_to_drop = [f for f in features_to_drop if f in df.columns]
    
    # Extract features (excluding label and unwanted columns)
    features = df.drop(columns=['label'] + existing_features_to_drop)
    labels = df['label']
    
    print(f"Original features: {len(df.columns)-1}")
    print(f"Features after dropping correlated ones: {len(features.columns)}")
    
    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # Dimensionality reduction with UMAP
    umap_model = umap.UMAP(
        n_components=CONFIG["umap_n_components"],
        random_state=CONFIG["random_state"]
    )
    features_umap = umap_model.fit_transform(features_scaled)
    
    return features, features_scaled, features_umap, labels, scaler

In [ ]:
features, features_scaled, features_umap, labels_true, scaler = prepare_features_for_clustering(final_feature_df)

## 4. Baseline Clustering (KMeans)

In [ ]:
def run_baseline_clustering(features_scaled, n_clusters=4):
    """Run baseline KMeans clustering."""
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=CONFIG["random_state"],
        n_init=10
    )
    kmeans_labels = kmeans.fit_predict(features_scaled)
    
    # Calculate metrics
    ari = adjusted_rand_score(labels_true, kmeans_labels)
    nmi = normalized_mutual_info_score(labels_true, kmeans_labels)
    silhouette = silhouette_score(features_scaled, kmeans_labels)
    
    print(f"Baseline KMeans Results:")
    print(f"  - Adjusted Rand Index (ARI): {ari:.4f}")
    print(f"  - Normalized Mutual Information (NMI): {nmi:.4f}")
    print(f"  - Silhouette Score: {silhouette:.4f}")
    
    return kmeans_labels, {"ARI": ari, "NMI": nmi, "Silhouette": silhouette}

In [ ]:
kmeans_labels, baseline_metrics = run_baseline_clustering(features_scaled, CONFIG["n_clusters"])

## 5. Sparse KMeans Implementation

In [ ]:
def soft_threshold(x, delta):
    """Soft-thresholding operator."""
    return np.sign(x) * np.maximum(np.abs(x) - delta, 0)

def sparse_kmeans(X, K, s, max_iter=100, tol=1e-6, patience=10, min_iter=5):
    """
    Sparse K-means clustering following Witten & Tibshirani (2010).
    
    Parameters:
    - X: numpy array of shape (n, p)
    - K: int, number of clusters
    - s: float, L1 bound on weights (1 <= s <= sqrt(p))
    - max_iter: int, maximum iterations
    - tol: float, convergence tolerance
    - patience: int, early stopping patience
    - min_iter: int, minimum iterations before early stopping
    
    Returns:
    - cluster_labels: array of cluster assignments
    - weights: array of feature weights
    - objectives: list of objective values per iteration
    """
    n, p = X.shape
    
    # Validate parameters
    if s < 1 or s > np.sqrt(p):
        raise ValueError(f"s must be between 1 and sqrt(p)={np.sqrt(p):.2f}")
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Weight initialization
    weights = np.ones(p) / np.sqrt(p)
    objective_history = []
    weight_history = [weights.copy()]

    # Early stopping variables
    best_weights = weights.copy()
    best_objective = -np.inf
    patience_counter = 0
    converged = False
    
    print(f"Sparse KMeans: {n} samples, {p} features, s={s}")
    
    for iteration in range(max_iter):
        # Weighted K-means clustering
        weighted_X = X_scaled * np.sqrt(weights)
        kmeans = KMeans(n_clusters=K, random_state=CONFIG["random_state"], n_init=10)
        cluster_labels = kmeans.fit_predict(weighted_X)
        
        # Compute BCSS (Between-Cluster Sum of Squares) for each feature
        BCSS = np.zeros(p)
        for j in range(p):
            overall_mean = np.mean(X_scaled[:, j])
            total_ss = np.sum((X_scaled[:, j] - overall_mean) ** 2)
            
            within_ss = 0
            for k in range(K):
                cluster_indices = (cluster_labels == k)
                n_k = np.sum(cluster_indices)
                if n_k > 0:
                    cluster_mean = np.mean(X_scaled[cluster_indices, j])
                    within_ss += np.sum((X_scaled[cluster_indices, j] - cluster_mean) ** 2)
            
            BCSS[j] = total_ss - within_ss
        
        # Update weights with L1 constraint
        a = BCSS / (np.linalg.norm(BCSS, 2) + 1e-10)
        
        # Binary search for delta to satisfy ||w||₁ = s
        def l1_norm(delta):
            w_temp = soft_threshold(a, delta)
            norm_w = np.linalg.norm(w_temp, 2)
            if norm_w > 0:
                w_temp = w_temp / norm_w
            return np.sum(np.abs(w_temp)) - s
        
        current_l1 = np.sum(np.abs(a / (np.linalg.norm(a, 2) + 1e-10)))
        
        if current_l1 <= s:
            delta = 0
            w_new = a
        else:
            delta_low, delta_high = 0, np.max(np.abs(a))
            try:
                delta = bisect(l1_norm, delta_low, delta_high, xtol=1e-8, maxiter=50)
                w_new = soft_threshold(a, delta)
            except:
                w_new = a
        
        # Normalize weights
        norm_w = np.linalg.norm(w_new, 2)
        if norm_w > 0:
            w_new = w_new / norm_w
        else:
            w_new = np.ones(p) / np.sqrt(p)
        
        # Calculate objective
        current_objective = np.sum(w_new * BCSS)
        objective_history.append(current_objective)
        weight_history.append(w_new.copy())

        # Check for improvement
        if current_objective > best_objective:
            best_objective = current_objective
            best_weights = w_new.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        # Check convergence
        weight_change = np.linalg.norm(w_new - weights, 2)
        n_nonzero = np.sum(w_new > 1e-6)
        
        if (iteration + 1) % 10 == 0:
            print(f"  Iteration {iteration + 1}: Obj={current_objective:.4f}, "
                  f"Non-zero={n_nonzero}/{p} ({n_nonzero/p*100:.1f}%)")
        
        # Early stopping conditions
        if weight_change < tol and iteration >= min_iter:
            print(f"Converged after {iteration + 1} iterations (weight change < {tol})")
            converged = True
            break
        elif patience_counter >= patience and iteration >= min_iter:
            print(f"Early stopping after {iteration + 1} iterations (no improvement for {patience} iterations)")
            converged = True
            break
            
        weights = w_new
    
    if not converged:
        print(f"Reached maximum iterations ({max_iter})")
    
    # Use best weights found
    final_weights = best_weights
    final_n_nonzero = np.sum(final_weights > 1e-6)
    
    # Run final clustering with best weights
    weighted_X_final = X_scaled * np.sqrt(final_weights)
    kmeans_final = KMeans(n_clusters=K, random_state=CONFIG["random_state"], n_init=10)
    final_labels = kmeans_final.fit_predict(weighted_X_final)
    
    print(f"Final: {final_n_nonzero} non-zero features, Best objective: {best_objective:.4f}")
    
    return final_labels, final_weights, objective_history

In [ ]:
# Run sparse KMeans
sparse_labels, sparse_weights, sparse_objectives = sparse_kmeans(
    X=features.values,
    K=CONFIG["n_clusters"],
    s=CONFIG["sparsity_s"],
    max_iter=50
)

## 6. Evaluation and Comparison

In [ ]:
def evaluate_clustering(true_labels, pred_labels, method_name):
    """Comprehensive evaluation of clustering results."""
    
    # Basic metrics
    ari = adjusted_rand_score(true_labels, pred_labels)
    nmi = normalized_mutual_info_score(true_labels, pred_labels)
    
    # Hungarian alignment for accuracy calculation
    def hungarian_remap(y_true, y_pred):
        labels_true = np.unique(y_true)
        labels_pred = np.unique(y_pred)
        cost_matrix = np.zeros((len(labels_true), len(labels_pred)), dtype=int)
        for i, true_label in enumerate(labels_true):
            for j, pred_label in enumerate(labels_pred):
                cost_matrix[i, j] = np.sum((y_true == true_label) & (y_pred != pred_label))
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        mapping = {labels_pred[col]: labels_true[row] for row, col in zip(row_ind, col_ind)}
        return np.vectorize(mapping.get)(y_pred)
    
    mapped_labels = hungarian_remap(true_labels, pred_labels)
    accuracy = np.mean(mapped_labels == true_labels)
    
    # Precision, recall, F1
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, mapped_labels, average='weighted'
    )
    
    print(f"{method_name} Results:")
    print(f"  - Adjusted Rand Index (ARI): {ari:.4f}")
    print(f"  - Normalized Mutual Information (NMI): {nmi:.4f}")
    print(f"  - Accuracy (after Hungarian alignment): {accuracy:.4f}")
    print(f"  - Precision: {precision:.4f}")
    print(f"  - Recall: {recall:.4f}")
    print(f"  - F1-Score: {f1:.4f}")
    
    return {
        "ARI": ari,
        "NMI": nmi,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

In [ ]:
# Evaluate baseline
print("=" * 60)
baseline_metrics_full = evaluate_clustering(labels_true, kmeans_labels, "Baseline KMeans")

# Evaluate sparse KMeans
print("=" * 60)
sparse_metrics = evaluate_clustering(labels_true, sparse_labels, "Sparse KMeans")

# Compare improvements
print("=" * 60)
print("Improvements (Sparse vs Baseline):")
for metric in ["ARI", "Accuracy", "F1"]:
    improvement = sparse_metrics[metric] - baseline_metrics_full[metric]
    print(f"  {metric}: {improvement:+.4f} "
          f"({baseline_metrics_full[metric]:.4f} → {sparse_metrics[metric]:.4f})")

## 7. Visualization

In [ ]:
def plot_cluster_comparison(features_2d, true_labels, baseline_labels, sparse_labels):
    """Plot clustering results comparison."""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # True labels
    unique_labels = np.unique(true_labels)
    for i, label in enumerate(unique_labels):
        mask = true_labels == label
        axes[0].scatter(features_2d[mask, 0], features_2d[mask, 1], 
                       label=label, s=20, alpha=0.7)
    axes[0].set_title("True Labels", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("UMAP 1")
    axes[0].set_ylabel("UMAP 2")
    axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Baseline KMeans
    scatter = axes[1].scatter(features_2d[:, 0], features_2d[:, 1], 
                             c=baseline_labels, cmap='tab10', s=20, alpha=0.7)
    axes[1].set_title(f"Baseline KMeans (ARI={baseline_metrics_full['ARI']:.3f})", 
                      fontsize=14, fontweight='bold')
    axes[1].set_xlabel("UMAP 1")
    axes[1].set_ylabel("UMAP 2")
    
    # Sparse KMeans
    scatter = axes[2].scatter(features_2d[:, 0], features_2d[:, 1], 
                             c=sparse_labels, cmap='tab10', s=20, alpha=0.7)
    axes[2].set_title(f"Sparse KMeans (ARI={sparse_metrics['ARI']:.3f})", 
                      fontsize=14, fontweight='bold')
    axes[2].set_xlabel("UMAP 1")
    axes[2].set_ylabel("UMAP 2")
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "cluster_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_feature_importance(weights, feature_names, top_n=20):
    """Plot feature importance from sparse KMeans."""
    
    # Sort features by weight
    sorted_indices = np.argsort(weights)[::-1]
    sorted_weights = weights[sorted_indices]
    sorted_names = np.array(feature_names)[sorted_indices]
    
    # Top N features
    top_weights = sorted_weights[:top_n]
    top_features = sorted_names[:top_n]
    
    plt.figure(figsize=(10, 8))
    bars = plt.barh(range(top_n), top_weights[::-1], color='steelblue')
    plt.yticks(range(top_n), top_features[::-1], fontsize=9)
    plt.xlabel("Feature Weight", fontsize=11)
    plt.title(f"Top {top_n} Features by Importance (Sparse KMeans)", fontsize=13, fontweight='bold')
    plt.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, (bar, weight) in enumerate(zip(bars, top_weights[::-1])):
        plt.text(weight + 0.001, bar.get_y() + bar.get_height()/2, 
                f"{weight:.3f}", ha='left', va='center', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Plot results
plot_cluster_comparison(features_umap, labels_true, kmeans_labels, sparse_labels)
plot_feature_importance(sparse_weights, features.columns.tolist(), top_n=20)

## 8. Save Results

In [ ]:
def save_results(final_feature_df, kmeans_labels, sparse_labels, sparse_weights, features):
    """Save clustering results and analysis."""
    
    # Create results dataframe
    results_df = final_feature_df.copy()
    results_df['kmeans_cluster'] = kmeans_labels
    results_df['sparse_kmeans_cluster'] = sparse_labels
    
    # Save to CSV
    results_df.to_csv(OUTPUT_DIR / "clustering_results.csv", index=False)
    
    # Save feature weights
    weights_df = pd.DataFrame({
        'feature': features.columns,
        'weight': sparse_weights
    }).sort_values('weight', ascending=False)
    weights_df.to_csv(OUTPUT_DIR / "feature_weights.csv", index=False)
    
    # Save metrics
    metrics_df = pd.DataFrame([
        {"Method": "Baseline KMeans", **baseline_metrics_full},
        {"Method": "Sparse KMeans", **sparse_metrics}
    ])
    metrics_df.to_csv(OUTPUT_DIR / "clustering_metrics.csv", index=False)
    
    # Save numpy arrays
    np.save(OUTPUT_DIR / "sparse_weights.npy", sparse_weights)
    np.save(OUTPUT_DIR / "features_umap.npy", features_umap)
    
    print(f"Results saved to {OUTPUT_DIR}/")
    print(f"  - clustering_results.csv")
    print(f"  - feature_weights.csv")
    print(f"  - clustering_metrics.csv")
    print(f"  - sparse_weights.npy")
    print(f"  - features_umap.npy")
    
    return results_df, weights_df, metrics_df

In [ ]:
results_df, weights_df, metrics_df = save_results(
    final_feature_df, kmeans_labels, sparse_labels, sparse_weights, features
)

## 9. Summary Report

In [ ]:
print("=" * 60)
print("CLUSTERING ANALYSIS SUMMARY")
print("=" * 60)

print(f"\nDataset Information:")
print(f"  - Samples: {len(final_feature_df)}")
print(f"  - Features after preprocessing: {len(features.columns)}")
print(f"  - Unique labels: {len(np.unique(labels_true))}")

print(f"\nClustering Performance:")
print(f"  Baseline KMeans:")
print(f"    • ARI: {baseline_metrics_full['ARI']:.4f}")
print(f"    • Accuracy: {baseline_metrics_full['Accuracy']:.4f}")
print(f"    • F1-Score: {baseline_metrics_full['F1']:.4f}")

print(f"\n  Sparse KMeans:")
print(f"    • ARI: {sparse_metrics['ARI']:.4f}")
print(f"    • Accuracy: {sparse_metrics['Accuracy']:.4f}")
print(f"    • F1-Score: {sparse_metrics['F1']:.4f}")
print(f"    • Non-zero features: {np.sum(sparse_weights > 1e-6)}/{len(sparse_weights)} "
      f"({np.sum(sparse_weights > 1e-6)/len(sparse_weights)*100:.1f}%)")

print(f"\nImprovements:")
for metric in ["ARI", "Accuracy", "F1"]:
    improvement = sparse_metrics[metric] - baseline_metrics_full[metric]
    percent_improvement = (improvement / baseline_metrics_full[metric]) * 100 if baseline_metrics_full[metric] != 0 else 0
    print(f"  • {metric}: {improvement:+.4f} ({percent_improvement:+.1f}%)")

print(f"\nTop 5 Most Important Features:")
top_features = weights_df.head(5)
for idx, row in top_features.iterrows():
    print(f"  {row['feature']}: {row['weight']:.4f}")

print(f"\nOutput Files:")
print(f"  All results saved to: {OUTPUT_DIR.absolute()}")

print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)